# Data Background

* https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily
* https://registry.opendata.aws/noaa-ghcn/

Global Historical Climatology Network - Daily is a dataset from NOAA that contains daily observations over global land areas

In [ ]:
!aws s3 ls s3://noaa-ghcn-pds/

In [ ]:
!aws s3 cp s3://noaa-ghcn-pds/readme.txt -

In [ ]:
!aws s3 cp s3://noaa-ghcn-pds/readme-by_year.txt -

In [ ]:
!aws s3 cp s3://noaa-ghcn-pds/ghcnd-stations.txt .

# Data Ingestion

Ingesting noaa weather data into bronze datalake in s3.

In [ ]:
# Dump data into s3
!aws s3 sync s3://noaa-ghcn-pds/parquet/by_year/ \
            s3://aebel-datalake/bronze/noaa_ghcn_daily/ \
            --exclude "*" --include "YEAR=20*" \
            --dryrun 

In [ ]:
!aws s3 ls s3://aebel-datalake/bronze/noaa_ghcn_daily/YEAR=2001/ELEMENT=PRCP/

In [ ]:
!aws s3 cp s3://noaa-ghcn-pds/ghcnd-stations.txt \
            s3://aebel-datalake/bronze/ghcnd_stations_raw/ \
            --dryrun 

# Investigation with polars

In [ ]:
import polars as pl
lf = pl.scan_parquet("s3://aebel-datalake/bronze/noaa_ghcn_daily/YEAR=2001/")
lf.collect_schema()

In [ ]:
lf.head().collect()

# Athena 


In [ ]:
import os
import polars as pl
from sqlalchemy import create_engine

os.environ["AWS_PROFILE"] = "default"      # or rely on env vars / SSO / instance role

REGION    = "eu-west-1"
WORKGROUP = "primary"
STAGING   = "s3://aebel-athena-results/results/"

engine = create_engine(
    f"awsathena+arrow://@athena.{REGION}.amazonaws.com:443"
    f"?s3_staging_dir={STAGING}&work_group={WORKGROUP}"
)

# also see pyproject.toml for magic sql config
pl.Config.set_tbl_rows(10)            # show up to 50 rows (default truncates to ~10 with "…")
pl.Config.set_fmt_str_lengths(200)    # don't truncate strings (default ~30 chars)
pl.Config.set_tbl_width_chars(250)    # allow wide tables before wrapping

%reload_ext sql
%sql engine

In [ ]:
%%sql
show tables from datalake_bronze

In [ ]:
%%sql
describe datalake_bronze.noaa_ghcn_daily

In [ ]:
%%sql
select distinct element from datalake_bronze.noaa_ghcn_daily where year=2001

In [ ]:
%%sql
-- show precipiation in india in 2001-07-05
SELECT *
FROM datalake_bronze.noaa_ghcn_daily
WHERE year = 2001
  AND element = 'PRCP'
  AND "date" = '20010705'
  AND id LIKE 'IN%'


In [ ]:
%%sql
select * from datalake_bronze.ghcnd_stations_raw 
where line like 'IN%KOZHIKODE%'

In [ ]:
%%sql
-- show precipiation in india in 2001-07-05
with indian_precipitation as (

SELECT *
FROM datalake_bronze.noaa_ghcn_daily
WHERE year = 2001
  AND element = 'PRCP'
  AND "date" = '20010705'
  AND id like 'IN%'
),

station_taxonomy as (
  select * from datalake_bronze.ghcnd_stations_raw 
where line like 'IN%'
)

select
  ip.id, ip.date, ip.data_value, st.line
from
  indian_precipitation ip
  inner join station_taxonomy st
  on st.line like ip.id || '%'